# Evaluation

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from wikifin_rag.rag_helper import RAGBase
from wikifin_rag.embedder import Embedder
from wikifin_rag.evaluation_utils import generate_corpus_ground_truth, evaluate
from wikifin_rag.config import PROJECT_ROOT
import pandas as pd
from psycopg import sql
import os

## Offline Evaluation Datasets
### Retrieval evaluation data

In [2]:
embedder = Embedder()

load_dotenv(override=True)
openai_client = OpenAI()

assistant = RAGBase(embedder=embedder, llm_client=openai_client)
db_client = assistant.db_client

In [3]:
def retrieve_documents(n=100):
    db_client.open_connection()
    
    db_client.cur.execute(
        sql.SQL(
            """
            SELECT
                c.document_id || '_' || c.chunk_id AS id,
                d.title,
                d.section,
                c.content
            FROM chunks c
            JOIN documents d
            ON c.document_id = d.id
            WHERE language = 'nl'
            LIMIT %s;
            """
        ),
        (n,)
    )
    documents = db_client.cur.fetchall()

    db_client.close_connection()

    return documents

In [4]:
# sets the location where the ground truth dataset is stored
dest = PROJECT_ROOT / "data"
filename = "ground_truth.csv"

# determines the size of the dataset
n_docs = 300 # number of documents to run the evaluation on
n_q = 5 # number of questions to generate per document

In [5]:
# TODO: set to True to rerun the data generation function
refresh_data = False

In [6]:
documents = retrieve_documents(n=n_docs)

In [7]:
if refresh_data or not os.path.exists(dest / filename):
    _, total_cost = generate_corpus_ground_truth(documents, llm_client=openai_client, n=n_q)

In [8]:
df_ground_truth = pd.read_csv(dest / filename)
ground_truth = df_ground_truth.to_dict(orient="records")

### RAG evaluation data

In [9]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [10]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["id"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [11]:
# answer_record = generate_rag_answer(ground_truth[0])

## Retrieval metrics
### Text Search

In [12]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [13]:
from wikifin_rag.evaluation_utils import compute_relevance, compute_relevance_total

In [14]:
db_client.open_connection()

In [15]:
def text_search(query, weights=None, normalization=0):
    return assistant.db_client.text_search(query=query, num_results=5, weights=weights, normalization=normalization)

In [16]:
query = ground_truth[3]["question"]
len(text_search(query))

5

In [17]:
ground_truth_sample = ground_truth[:15]

In [18]:
def objective(args):
    normalization = args["normalization"]
    del args["normalization"]

    retrieval_metrics = evaluate(
        ground_truth_sample,
        lambda query: text_search(query=query, weights=args, normalization=normalization))

    return {'loss': -retrieval_metrics['mrr'], 'status': STATUS_OK }

In [19]:
search_space = {
    "A": hp.loguniform("A", -5, 0),
    "B": hp.loguniform("B", -7, 0),
    "C": hp.loguniform("C", -3, 0),
    "D": 1 - hp.uniform("D_raw", 0, 1) ** 2,
    "normalization": hp.choice("normalization", [0, 1, 2, 4, 8, 16, 32])
}

In [20]:
best_params = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials(),
    show_progressbar=False,
    trials_save_file=PROJECT_ROOT / "data/text_search_trials.pkl"
)

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

...

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

In [21]:
best

{'A': np.float64(0.985682406477342),
 'B': np.float64(0.003599489592177068),
 'C': np.float64(0.2031240515189807),
 'D_raw': np.float64(0.2192674347256001),
 'normalization': np.int64(6)}

### Vector Search

In [22]:
def vector_search(query):
    return assistant.db_client.vector_search(query=query, num_results=5)

### Hybrid Search

In [23]:
def hybrid_search(query):
    return assistant.search(query=query, num_results=5)

In [24]:
db_client.close_connection()

In [33]:
trials = pd.read_pickle(PROJECT_ROOT / "data/text_search_trials.pkl")

In [42]:
trials.trials[0]

{'state': 2,
 'tid': 0,
 'spec': None,
 'result': {'loss': -0.4867, 'status': 'ok'},
 'misc': {'tid': 0,
  'cmd': ('domain_attachment', 'FMinIter_Domain'),
  'workdir': None,
  'idxs': {'A': [np.int64(0)],
   'B': [np.int64(0)],
   'C': [np.int64(0)],
   'D_raw': [np.int64(0)],
   'normalization': [np.int64(0)]},
  'vals': {'A': [np.float64(0.05017628228428399)],
   'B': [np.float64(0.7880207966602142)],
   'C': [np.float64(0.6215308310475072)],
   'D_raw': [np.float64(0.35101668995650137)],
   'normalization': [np.int64(0)]}},
 'exp_key': None,
 'owner': None,
 'version': 0,
 'book_time': datetime.datetime(2026, 8, 27, 7, 44, 44, 636000),
 'refresh_time': datetime.datetime(2026, 8, 27, 7, 44, 50, 663000)}